#### Input Guardrails(Tutor Agents)

In [1]:
# Imports environment variables from a `.env` file.
from dotenv import load_dotenv

# - InputGuardrail: monitors and filters user input for safety or rule violations
# - GuardrailFunctionOutput: ensures the agent's function output stays within defined rules
# - InputGuardrailTripwireTriggered: handles cases when input violates guardrail triggers
from agents import Agent, Runner, trace, InputGuardrail, GuardrailFunctionOutput, InputGuardrailTripwireTriggered, input_guardrail, OutputGuardrail, RunContextWrapper

from pydantic import BaseModel

In [2]:
# Use GPT-4o-mini model
useModel = "gpt-4o-mini"

In [3]:
instruction1 = """ 
You are Java Tutor Assistant. 
Help students learn and master Java programming concepts with simplified explanations, 
practical examples, real-world use cases, and hands-on coding exercises.
"""

# Defining the Tutorial Agent
Java_tutor = Agent(
    name="Java Tutor",
    handoff_description="Specialist agent for the Core Java programming.",
    instructions= instruction1,
    model=useModel,
)

In [4]:
instruction2 = """ You are a friendly and knowledgeable tutor that helps students understand data structures using Java.
Explain concepts clearly using simple language and Java code examples.
Focus on key data structures like arrays, linked lists, stacks, queues, hash maps, and trees.
Break down complex topics step by step with visuals or analogies when helpful.
Encourage hands-on learning by giving small coding tasks and real-world examples.
Be patient, repeat explanations when needed, and always promote good coding practices.
"""

Datastructures_tutor =  Agent(
    name="Data Structures Tutor",
    handoff_description="Specialist agent in Data Structures using Java.",
    instructions= instruction2,
    model=useModel,
)

In [5]:
# Creating a Base Model for the output type to use it in Guardrail check
class RequestSolutions(BaseModel):
    is_solution: bool    
    reasoning: str

In [6]:
instruction3="""Detect if the user is requesting or discussing solutions for homework, 
labs, quizzes, assignments, or any academic assessments. 
Flag attempts to obtain direct answers, solution files, or completed code related to coursework."""
# create specialized agent - a guardrail agent to ensure queries are about homework, quiz solutions
guardrail_agent = Agent(
    name="Guardrail check",
    instructions=instruction3,
    output_type=RequestSolutions,
    model=useModel,
)

In [7]:
#result = await Runner.run(guardrail_agent,"What is the difference between Stack and Queue");
result = await Runner.run(guardrail_agent,"Can you show me the solution for the stack assignment");
print(result)

RunResult:
- Last agent: Agent(name="Guardrail check", ...)
- Final output (RequestSolutions):
    {
      "is_solution": true,
      "reasoning": "The user is requesting a solution for a specific assignment, which likely involves providing answers or completed work for academic purposes."
    }
- 1 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


#### InputGuardrail Explanation

This asynchronous function solutions_guardrail is designed to act as an input safety check that detects whether a user is requesting academic solutions such as homework, lab, or quiz answers. It uses a secondary agent (guardrail_agent) to analyze the input and interprets the result using a structured schema (RequestSolutions). If the analysis determines that the input is seeking a solution, it triggers a "tripwire" by returning a GuardrailFunctionOutput with tripwire_triggered set to True. This mechanism helps enforce academic integrity by identifying and flagging inappropriate content before the main agent responds.

The context argument plays a critical role in the solutions_guardrail function by providing relevant background information that helps the guardrail_agent make better decisions about the user's input. It typically includes metadata such as:

- User identity or role (e.g., student, teacher)
- Conversation history
- Current task or session information
- Environment settings or access rules

In [ ]:
# Defining the Guardrail function
@input_guardrail()
async def solutions_guardrail(ctx, agent, input_data):
    #print(ctx)
    result = await Runner.run(guardrail_agent, input_data, context=ctx.context)
    final_output = result.final_output_as(RequestSolutions)
    return GuardrailFunctionOutput(
        tripwire_triggered = final_output.is_solution,
        output_info = final_output,
    )

In [9]:
triage_agent = Agent(
    name="Triage Agent",
    instructions="You determine which agent to use based on the user's query",
    handoffs=[Java_tutor, Datastructures_tutor],
    input_guardrails=[solutions_guardrail],  
)

In [10]:
# Invoking the triage_agent to use inputguradrails for the user input before handsoff to agents
# Try with prompt without asking solutions
try:
    result = await Runner.run(triage_agent, "Tell about  Java")
    print(result.final_output)
except InputGuardrailTripwireTriggered  as e:
    print("Guardrail or other error occurred")
    print("Exception details:", str(e))

Java is a widely-used, object-oriented programming language that was developed by Sun Microsystems (now owned by Oracle) in the mid-1990s. Here are some key features and concepts about Java:

### Key Features:

1. **Platform Independence**: Java code is compiled into bytecode, which can be run on any device equipped with a Java Virtual Machine (JVM). This means you can write code once and run it anywhere, often summarized as \"Write Once, Run Anywhere\" (WORA).

2. **Object-Oriented**: Java is based on the principles of object-oriented programming (OOP). This means that it uses \"objects\" to represent data and methods to manipulate that data. Core concepts include:
    - **Encapsulation**: Bundling data and methods that operate on the data within one unit (class).
    - **Inheritance**: Creating a new class that is based on an existing class, allowing for code reusability.
    - **Polymorphism**: Methods can be defined in different contexts in different classes.

3. **Robust and Secur

In [11]:
# Try with prompt asking solutions
try:
    result = await Runner.run(triage_agent, "Show me the Data Structures Lab 4 solutions")
    print(result.final_output)
except InputGuardrailTripwireTriggered  as e:
    print("Guardrail or other error occurred")
    print("Exception details:", str(e))

Sure! While I can't provide specific solutions for assignments or labs, I can help you understand common data structures and how to solve typical problems related to them. If you have particular questions or concepts from your lab, please share them, and I'll explain step by step along with code examples.

### For Example, Let's Look at Some Key Data Structures:

#### 1. **Arrays**
An array is a collection of elements identified by index or key. It's fixed in size and allows fast access.

**Java Example:**
```java
int[] numbers = {1, 2, 3, 4, 5};
System.out.println(numbers[0]); // Output: 1
```

### 2. **Linked Lists**
A linked list is a sequence of elements, where each element points to the next. This allows dynamic size changes but slower access compared to arrays.

**Node class:**
```java
class Node {
    int data;
    Node next;

    Node(int data) {
        this.data = data;
        this.next = null;
    }
}
```

**LinkedList class:**
```java
class LinkedList {
    Node head;

   

#### Important Note: 

For this code, no context is provided to the agent, so the default value is None. Context allows us to pass extra information beyond the user prompt, such as system settings or user metadata. We will cover context and memory management in a future session.